# Geração do DoE de um cenário — passo a passo

> **Notebook-irmão de [`scripts/generate_doe.py`](../scripts/generate_doe.py).** Cumpre o mesmo papel do
> script — do YAML do cenário ao cockpit `.xlsx` que o ProModel percorre — chamando **as mesmas
> funções do pacote**, só que uma de cada vez, com a explicação de cada passo e a inspeção do que
> cada uma devolve. Em resumo, ele:
>
> 1. carrega e valida o YAML do cenário (`sed_meta.config`);
> 2. gera o design LHS (critério ESE) e arredonda os tempos a 1 casa;
> 3. expande em plano de execução: 300 pontos × 3 réplicas = 900 execuções;
> 4. monta e grava o cockpit no layout padrão do ProModel;
> 5. relê o arquivo e confere o contrato (900 pendentes, fatores constantes dentro do ponto,
>    limites respeitados);
> 6. mostra a tabela que o script espelha na aba `scenario N doe` do `results-by-test.xlsx`.
>
> **Diferença deliberada:** este notebook grava em `Código/runs/notebook_doe_<DoE-ID>/` e **nunca**
> na pasta `promodel_interface/` nem no `results-by-test.xlsx`. Quem produz o cockpit oficial é o
> script, que grava o registro (cockpit + `doe_metadata_<DoE-ID>.json`) em
> `Código/runs/doe_scenario_0N_<DoE-ID>/` e copia **só o cockpit** para a interface:
>
> ```bash
> cd Código
> ../venv/Scripts/python.exe scripts/generate_doe.py configs/scenario_01_serial.yaml
> ```

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ_CODIGO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ_CODIGO / "src"))

# Troque aqui para gerar outro cenário
CENARIO = "scenario_01_serial"
CONFIG_PATH = RAIZ_CODIGO / "configs" / f"{CENARIO}.yaml"
pd.set_option("display.max_columns", 30)

## 1. O YAML do cenário é a fonte de verdade

`load_scenario_config()` lê o YAML e o valida com pydantic (`ScenarioConfig`). Campo digitado
errado, limite invertido (`low >= high`), `decimals` num fator inteiro, nome de coluna repetido ou
colidindo com `Item`/`point_id`/`replication_id`/`unique_id` — tudo isso **falha aqui**, antes de qualquer amostragem.

A ordem de `factors` e `responses` no YAML é a ordem das colunas no cockpit: a Macro do ProModel
lê por posição. `target` fica nulo: o projeto prevê **um KPI por estudo**, escolhido pelo usuário.

In [2]:
from sed_meta.config import load_scenario_config

config = load_scenario_config(CONFIG_PATH)
print(config.scenario.name, "—", config.scenario.description)
print("DoE        :", config.doe)
print("Réplicas   :", config.replications)
print("Layout     :", config.cockpit)
print("Target     :", config.target)

pd.DataFrame([f.model_dump() for f in config.factors]).set_index("name")

scenario_01_serial — Linha de produção serial P1 -> P2 -> P3 (baseline)
DoE        : method='lhs' criterion='ese' n_points=300 seed=2611101
Réplicas   : n_replications=3
Layout     : sheet_name='cockpit' item_column='Item' item_prefix='Teste '
Target     : None


,type,low,high,levels,decimals,unit,rationale
name,,,,,,,
Capacidade M1,integer,1.0,3.0,None,NaN,peças simultâneas,peças que a locação processa ao mesmo tempo; 1...
Quantidade chegada,integer,1.0,2.0,None,NaN,peças/chegada,peças por evento de chegada
Ocorrências chegada,integer,50.0,150.0,None,NaN,chegadas,número de eventos de chegada na rodada
Frequência chegada,float,4.0,16.0,None,1.0,min,intervalo entre chegadas; 4–16 min (com 2–8 a ...
Tempo processo 1,float,2.0,10.0,None,1.0,min,tempo de processamento por peça
Capacidade M2,integer,1.0,3.0,None,NaN,peças simultâneas,peças que a locação processa ao mesmo tempo; 1...
Capacidade M3,integer,1.0,3.0,None,NaN,peças simultâneas,peças que a locação processa ao mesmo tempo; 1...
Tempo processo 2,float,2.0,10.0,None,1.0,min,tempo de processamento por peça
Tempo processo 3,float,2.0,10.0,None,1.0,min,tempo de processamento por peça


In [3]:
pd.DataFrame([r.model_dump() for r in config.responses]).set_index("name").T

name,Entradas,Prontas,Tempo_ate_P1,Tempo_ate_P2,Tempo_ate_P3,Tempo_P1,Tempo_P2,Tempo_P3,Feito_P1,Feito_P2,Feito_P3,Tempo_total,Tempo_simul
unit,peças,peças,s,s,s,s,s,s,peças,peças,peças,s,s
description,peças que entraram no sistema,peças que saíram prontas,tempo até a peça chegar ao processo 1,tempo até a peça chegar ao processo 2,tempo até a peça chegar ao processo 3,tempo no processo 1,tempo no processo 2,tempo no processo 3,peças processadas em P1,peças processadas em P2,peças processadas em P3,tempo total da peça no sistema (lead time),duração simulada da rodada


## 2. O design: `generate_design()` sobre os `ParameterSpec`

`to_parameter_specs()` traduz os fatores para a entrada de `parameter_space.generate_design()`,
que amostra o cubo unitário com o LHS do SMT (critério ESE) e decodifica para os limites
declarados. Inteiros usam `floor(u · n_níveis)`, o que dá a todos os níveis — inclusive os das
bordas — a mesma probabilidade.

O Sobol foi descartado para esta campanha: 300 não é potência de 2, e com outro $n$ a sequência
perde o balanceamento que justifica usá-la.

In [4]:
import time
from sed_meta.doe.parameter_space import generate_design

specs = config.to_parameter_specs()
t0 = time.perf_counter()
design_continuo = generate_design(specs, n_points=config.doe.n_points, seed=config.doe.seed,
                                  method=config.doe.method, criterion=config.doe.criterion)
print(f"{len(design_continuo)} pontos x {design_continuo.shape[1]} fatores "
      f"em {time.perf_counter() - t0:.1f} s")
design_continuo.head()

300 pontos x 9 fatores em 1.8 s


,Capacidade M1,Quantidade chegada,Ocorrências chegada,Frequência chegada,Tempo processo 1,Capacidade M2,Capacidade M3,Tempo processo 2,Tempo processo 3
0,2,1,144,12.529544,3.783937,3,3,8.732246,5.563836
1,2,2,119,4.171588,7.656653,3,2,3.088370,4.807232
2,2,1,51,7.970661,2.590303,3,3,7.956357,7.002964
3,3,1,100,8.965068,2.550959,3,3,6.588581,3.942563
4,2,1,69,15.640114,4.668711,2,2,7.118318,6.625421


## 3. Arredondamento: `round_design()`

Os tempos e a frequência de chegada vão ao cockpit com **1 casa decimal** (`decimals: 1` no YAML).
O arredondamento é feito **depois** do gerador, em `sed_meta.config`, e não dentro de
`parameter_space`: o LHS continua estratificando o contínuo, e a resolução com que o
operador/ProModel enxerga o valor é decisão do cenário. O dtype continua `float64`.

Se a variável do ProModel que recebe o tempo for inteira, os decimais seriam truncados — nesse caso
o certo é mudar o fator para `type: integer` no YAML (pendência registrada no
`context-for-claude.md`).

In [5]:
from sed_meta.config import round_design

design = round_design(design_continuo, config)
floats = [f.name for f in config.factors if f.decimals is not None]
print("maior ajuste do arredondamento:", float((design[floats] - design_continuo[floats]).abs().max().max()))
print("dtypes:", design.dtypes.astype(str).value_counts().to_dict())
design.describe().T[["min", "max", "mean"]]

maior ajuste do arredondamento: 0.04999288767242316
dtypes: {'int64': 5, 'float64': 4}


,min,max,mean
Capacidade M1,1.0,3.0,2.000000
Quantidade chegada,1.0,2.0,1.500000
Ocorrências chegada,50.0,150.0,99.993333
Frequência chegada,4.0,16.0,9.998333
Tempo processo 1,2.0,10.0,6.002000
Capacidade M2,1.0,3.0,2.000000
Capacidade M3,1.0,3.0,2.000000
Tempo processo 2,2.0,10.0,6.000333
Tempo processo 3,2.0,10.0,6.000333


### 3.1 O arredondamento estragou o preenchimento?

`fill_metrics()` (as mesmas métricas da Seção 4.1 do `pipeline.ipynb`, agora no pacote) é calculada
no cubo normalizado pelos limites **declarados**. Comparar o design contínuo com o arredondado
mostra que 1 casa decimal sobre faixas de 6–8 minutos praticamente não muda nada. O script grava
as duas colunas no `doe_metadata_<id>.json`, em `Código/runs/`.

In [6]:
from sed_meta.doe.parameter_space import fill_metrics

pd.DataFrame({"contínuo": fill_metrics(design_continuo, specs),
              "arredondado": fill_metrics(design, specs)}).round(5)

,contínuo,arredondado
dist_min_entre_pontos,0.41108,0.40984
dist_media_ao_vizinho,0.65994,0.66016
desvio_dist_ao_vizinho,0.07462,0.07499
dist_media_entre_pares,1.50115,1.50127
cobertura_dos_vertices,0.79443,0.79453
discrepancia_L2_estrela,0.00521,0.00521
disp_espacamento_marginal,0.02112,0.02282
pontos_duplicados,0.00000,0.00000
dimensoes,9.00000,9.00000


## 4. Plano de execução: `build_execution_plan()`

Cada ponto do design vira `n_replications` linhas, com `point_id` (o ponto de origem),
`replication_id` (0..R−1 dentro do ponto) e um `unique_id` (id único da execução). **Uma linha = uma execução
do ProModel.** A expansão é vetorizada para preservar dtypes.

In [7]:
from sed_meta.campaign import build_plan_and_cockpit

plan, cockpit = build_plan_and_cockpit(design, config)
print("plano:", plan.shape, "| cockpit:", cockpit.shape)
plan.head(6)

plano: (900, 12) | cockpit: (900, 25)


,point_id,replication_id,unique_id,Capacidade M1,Quantidade chegada,Ocorrências chegada,Frequência chegada,Tempo processo 1,Capacidade M2,Capacidade M3,Tempo processo 2,Tempo processo 3
0,0,0,0,2,1,144,12.5,3.8,3,3,8.7,5.6
1,0,1,1,2,1,144,12.5,3.8,3,3,8.7,5.6
2,0,2,2,2,1,144,12.5,3.8,3,3,8.7,5.6
3,1,0,3,2,2,119,4.2,7.7,3,2,3.1,4.8
4,1,1,4,2,2,119,4.2,7.7,3,2,3.1,4.8
5,1,2,5,2,2,119,4.2,7.7,3,2,3.1,4.8


`build_cockpit_table()` (chamada dentro de `build_plan_and_cockpit`) só acrescenta à direita uma
coluna `float64` vazia por saída. Célula vazia é o marcador de **"ainda não executado"**.

In [8]:
print("células de saída preenchidas:", int(cockpit[config.response_names].notna().to_numpy().sum()))
cockpit.iloc[:3, -5:]

células de saída preenchidas: 0


,Feito_P1,Feito_P2,Feito_P3,Tempo_total,Tempo_simul
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN


## 5. O cockpit no layout do ProModel: `write_promodel_cockpit()`

O padrão visual está no código (`excel_bridge`, constantes `_PM_*`) — nenhum arquivo-modelo é
lido, e todo cenário usa o mesmo padrão:

| linha | conteúdo |
|---|---|
| 1 | vazia |
| 2 | faixas mescladas: `Entrada` (verde, de `Item` à última entrada), `Saída` (vermelha), `Identificação` (cinza) |
| 3 | (A vazia) \| `Item` \| entradas \| saídas \| `point_id` \| `replication_id` \| `unique_id` |
| 4+ | \| `Teste n` (azul-marinho) \| valores (amarelos) \| saídas vazias (amarelas) \| ids (cinza claro) |

- `Item` = `Teste n`, com n = 1..900 = índice da execução: é o **ponteiro da Macro** do ProModel.
- `point_id` (constante entre as réplicas do ponto), `replication_id` e `unique_id` ficam **à
  direita das saídas**, com cor própria, para não deslocar o que a Macro lê.
- A aba oculta `_sed_meta_schema` guarda coluna, papel e dtype, para a volta restaurar os tipos.

O notebook grava em `Código/runs/`, nunca na interface.

In [9]:
from sed_meta.campaign import cockpit_filename, new_doe_id, write_cockpit

DOE_ID = new_doe_id()
OUT_DIR = RAIZ_CODIGO / "runs" / f"notebook_doe_{DOE_ID}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CAMINHO = OUT_DIR / cockpit_filename(config, DOE_ID)

write_cockpit(cockpit, CAMINHO, config)
print("gravado:", CAMINHO.relative_to(RAIZ_CODIGO))

gravado: runs\notebook_doe_261103-0006\cockpit_scenario_01_lhs_300x3_261103-0006.xlsx


Como o arquivo ficou, lido célula a célula (Item, as primeiras entradas e as colunas de identificação):

In [10]:
import openpyxl

ws = openpyxl.load_workbook(CAMINHO)["cockpit"]
print("faixas mescladas:", sorted(str(r) for r in ws.merged_cells.ranges))
print("abas:", [(s.title, s.sheet_state) for s in ws.parent.worksheets])
colunas = [*range(2, 6), *range(ws.max_column - 2, ws.max_column + 1)]
pd.DataFrame([[ws.cell(r, j).value for j in colunas] for r in range(2, 10)],
             index=[f"linha {r}" for r in range(2, 10)],
             columns=[openpyxl.utils.get_column_letter(j) for j in colunas])

faixas mescladas: ['B2:K2', 'L2:X2', 'Y2:AA2']
abas: [('cockpit', 'visible'), ('_sed_meta_schema', 'hidden')]


,B,C,D,E,Y,Z,AA
linha 2,Entrada,None,None,None,Identificação,None,None
linha 3,Item,Capacidade M1,Quantidade chegada,Ocorrências chegada,point_id,replication_id,unique_id
linha 4,Teste 1,2,1,144,0,0,0
linha 5,Teste 2,2,1,144,0,1,1
linha 6,Teste 3,2,1,144,0,2,2
linha 7,Teste 4,2,2,119,1,0,3
linha 8,Teste 5,2,2,119,1,1,4
linha 9,Teste 6,2,2,119,1,2,5


## 6. Volta: `read_promodel_cockpit()` + `verify_cockpit()`

`read_promodel_cockpit()` devolve o arquivo no vocabulário interno (`point_id`, `replication_id`,
`unique_id`, parâmetros, KPIs) com os dtypes restaurados — a saída vai direto para
`extract_results()`/`select_complete_runs()`, sem adaptação. É a mesma função que vai ler o
cockpit devolvido pelo ProModel em `outputs_of_simulation/`.

`verify_cockpit()` é a conferência que o script faz antes de declarar o arquivo pronto.

In [11]:
from sed_meta.campaign import verify_cockpit
from sed_meta.sim_interface.excel_bridge import extract_results, read_promodel_cockpit

volta = read_promodel_cockpit(str(CAMINHO))
print(volta.dtypes.astype(str).value_counts().to_dict())
raw = extract_results(volta, config.response_names)
print("status:", raw["status_execucao"].value_counts().to_dict())

verify_cockpit(CAMINHO, config, plan)

{'float64': 17, 'int64': 8}
status: {'pendente': 900}


{'linhas': 900,
 'status': {'pendente': 900},
 'point_ids': 300,
 'replicas_por_ponto': 3,
 'fatores_variando_dentro_do_ponto': 0,
 'valores_fora_dos_limites': 0,
 'dtypes': {'Capacidade M1': 'int64',
  'Quantidade chegada': 'int64',
  'Ocorrências chegada': 'int64',
  'Frequência chegada': 'float64',
  'Tempo processo 1': 'float64',
  'Capacidade M2': 'int64',
  'Capacidade M3': 'int64',
  'Tempo processo 2': 'float64',
  'Tempo processo 3': 'float64'}}

## 7. O espelho no `results-by-test.xlsx`

O script reescreve a aba `scenario N doe` com esta tabela (mais um bloco com método, pontos,
réplicas, seed do DoE e DoE-ID). O YAML é a fonte de verdade; a aba é só espelho. Aqui só mostramos —
o notebook não escreve na planilha de registro.

In [12]:
from sed_meta.campaign import doe_notes_rows

pd.DataFrame(doe_notes_rows(config),
             columns=["Coluna", "Papel", "Tipo", "Unidade", "Low", "High", "Casas", "Justificativa"])

,Coluna,Papel,Tipo,Unidade,Low,High,Casas,Justificativa
0,Item,id,texto,,NaN,NaN,NaN,"'Teste n', n = 1..N = índice da execução (pont..."
1,Capacidade M1,entrada,integer,peças simultâneas,1.0,3.0,NaN,peças que a locação processa ao mesmo tempo; 1...
2,Quantidade chegada,entrada,integer,peças/chegada,1.0,2.0,NaN,peças por evento de chegada
3,Ocorrências chegada,entrada,integer,chegadas,50.0,150.0,NaN,número de eventos de chegada na rodada
4,Frequência chegada,entrada,float,min,4.0,16.0,1.0,intervalo entre chegadas; 4–16 min (com 2–8 a ...
5,Tempo processo 1,entrada,float,min,2.0,10.0,1.0,tempo de processamento por peça
6,Capacidade M2,entrada,integer,peças simultâneas,1.0,3.0,NaN,peças que a locação processa ao mesmo tempo; 1...
7,Capacidade M3,entrada,integer,peças simultâneas,1.0,3.0,NaN,peças que a locação processa ao mesmo tempo; 1...
8,Tempo processo 2,entrada,float,min,2.0,10.0,1.0,tempo de processamento por peça
9,Tempo processo 3,entrada,float,min,2.0,10.0,1.0,tempo de processamento por peça


## Próximo passo

O cockpit oficial (gerado pelo script, com cópia em
`promodel_interface/inputs_to_simulation/DoE_cockpits_scenario_0N/`) vai para o ProModel. O arquivo
devolvido, com as saídas preenchidas, entra em
`promodel_interface/outputs_of_simulation/simulation_results_scenario_0N/` e é lido com
`read_promodel_cockpit()` → `extract_results()` → `select_complete_runs()`.